# ⚡ Unleashed AI — Combined Server

Your **Dolphin-Llama3 GGUF from Google Drive** + **SDXL Turbo** images.
One notebook, one ngrok tunnel.

**Setup:** Runtime → Change runtime type → **T4 GPU** → Run all cells.

In [ ]:
#@title 1️⃣ Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q llama-cpp-python==0.2.77 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q diffusers transformers accelerate safetensors
!pip install -q fastapi uvicorn pyngrok
print("✅ Drive mounted & dependencies installed")

In [ ]:
#@title 2️⃣ Find & Load Your GGUF Model
import os

# Find your gguf file
print("Searching for .gguf files...")
!find /content/drive/MyDrive -name "*.gguf" 2>/dev/null
print("\n👆 Copy the path above and paste it below if the auto-detect doesn't work")

In [ ]:
#@title 3️⃣ Load Text Model from Drive
import torch, glob
from llama_cpp import Llama

# Auto-detect the gguf file, or paste your path manually
gguf_files = glob.glob("/content/drive/MyDrive/**/*.gguf", recursive=True)
if gguf_files:
    model_path = gguf_files[0]
    print(f"Found: {model_path}")
else:
    # Fallback — paste your path here:
    model_path = "/content/drive/MyDrive/models/YOUR_MODEL.gguf"  #@param {type:"string"}
    print(f"Using manual path: {model_path}")

print(f"Size: {os.path.getsize(model_path)/1e9:.1f}GB")

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=2,
    verbose=False
)
print("✅ Text model loaded")

In [ ]:
#@title 4️⃣ Load SDXL Turbo (no safety checker)
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to("cuda")
pipe.safety_checker = None
pipe.requires_safety_checker = False
pipe.enable_attention_slicing()

test = pipe("a red cube", num_inference_steps=1, guidance_scale=0.0).images[0]
print(f"✅ SDXL Turbo loaded — Total VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB / 15GB")
test

In [ ]:
#@title 5️⃣ Start API Server (one tunnel for chat + images)

import io, base64, threading, time, json
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
from pyngrok import ngrok

app = FastAPI(title="Unleashed AI")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: Optional[str] = "dolphin-llama3"
    messages: List[Message]
    stream: Optional[bool] = False
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = -1

class ImageRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "ugly, blurry, low quality, deformed"
    width: Optional[int] = 512
    height: Optional[int] = 512
    steps: Optional[int] = 4
    guidance_scale: Optional[float] = 0.0

@app.get("/health")
def health():
    return {"status": "ok", "models": ["dolphin-llama3", "sdxl-turbo"], "gpu": torch.cuda.get_device_name(0)}

@app.get("/v1/models")
def models():
    return {"data": [{"id": "dolphin-llama3", "object": "model"}]}

@app.post("/v1/chat/completions")
def chat_completions(body: ChatRequest):
    messages = [{"role": m.role, "content": m.content} for m in body.messages]
    max_tok = body.max_tokens if body.max_tokens and body.max_tokens > 0 else 2048

    if body.stream:
        def generate():
            stream = llm.create_chat_completion(
                messages=messages, max_tokens=max_tok,
                temperature=body.temperature, stream=True
            )
            for chunk in stream:
                delta = chunk["choices"][0].get("delta", {})
                finish = chunk["choices"][0].get("finish_reason")
                data = {
                    "id": "chatcmpl-unleashed",
                    "object": "chat.completion.chunk",
                    "model": "dolphin-llama3",
                    "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]
                }
                yield f"data: {json.dumps(data)}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(generate(), media_type="text/event-stream")
    else:
        return llm.create_chat_completion(messages=messages, max_tokens=max_tok, temperature=body.temperature)

@app.post("/generate")
def generate_image(req: ImageRequest):
    try:
        start = time.time()
        w = (min(max(req.width, 256), 768) // 8) * 8
        h = (min(max(req.height, 256), 768) // 8) * 8
        image = pipe(
            prompt=req.prompt, negative_prompt=req.negative_prompt,
            num_inference_steps=req.steps, guidance_scale=req.guidance_scale,
            width=w, height=h,
        ).images[0]
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        img_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        elapsed = time.time() - start
        print(f"🎨 {elapsed:.1f}s: {req.prompt[:60]}")
        return {"image": img_b64, "elapsed": elapsed}
    except Exception as e:
        print(f"❌ {e}")
        return {"error": str(e)}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
time.sleep(2)

ngrok.set_auth_token("39phl7sSqS6GFghufY5fOWPXoSM_62bNCvdb1MAG1K4xkaitY")
public_url = ngrok.connect(8000)

print("\n" + "="*60)
print(f"⚡ UNLEASHED AI — SERVER READY")
print(f"")
print(f"   🌐 {public_url}")
print(f"")
print(f"   Paste into server/index.js line 18:")
print(f"   const COLAB_URL = \"{public_url}\";")
print(f"")
print(f"   🧠 Chat:   /v1/chat/completions")
print(f"   🎨 Images: /generate")
print(f"   📖 Docs:   {public_url}/docs")
print("="*60)

In [ ]:
#@title 6️⃣ Keep Alive
import time
print(f"⏳ Server running at: {public_url}")
try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\n🛑 Stopped")